# Week 3, Lab 5 — Mini-project: research, draft, review


In [ ]:
WEEK = 'Week 3'
LAB = 'Lab 5 — mini-project'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn crewai
else:
    %pip install -q crewai ollama


In [ ]:
cfg = openai_client_kwargs()
from crewai import LLM, Agent, Task, Crew, Process

llm = LLM(
    model=f"openai/{cfg['model']}",
    api_key=cfg["api_key"],
    base_url=cfg["base_url"],
)
print("CrewAI LLM ->", cfg)


In [ ]:
from crewai.tools import BaseTool

class LookupTool(BaseTool):
    name: str = "lookup_fact"
    description: str = "Local KB lookup."
    def _run(self, topic: str) -> str:
        return lookup_fact(topic)

researcher = Agent(role="Researcher", goal="Gather facts with the tool.", backstory="Analyst.", llm=llm, tools=[LookupTool()])
writer = Agent(role="Writer", goal="Draft a 120-word student explainer.", backstory="Teacher.", llm=llm)
reviewer = Agent(role="Reviewer", goal="Check factuality vs the research notes; return a corrected final draft.", backstory="Strict editor.", llm=llm)

topic = "MCP"
t1 = Task(description=f"Look up facts about {topic} and CrewAI.", expected_output="Bullets from tools.", agent=researcher)
t2 = Task(description="Draft ~120 words for beginners.", expected_output="One short essay.", agent=writer)
t3 = Task(description="Review and produce the FINAL student-facing text only.", expected_output="Final draft.", agent=reviewer)
print(Crew(agents=[researcher, writer, reviewer], tasks=[t1, t2, t3], process=Process.sequential).kickoff())


## Rubric

Three roles, at least one tool call, a reviewer pass, no paid API key.
